# Accelerator compiler support visualizations

This notebook reads the committed compatibility snapshots and exports four publication/documentation PDFs:

1. inventory funnel and compilation-coverage bars;
2. three-platform support-overlap diagram;
3. operator-family coverage heatmap;
4. multipage, function-level compatibility atlas.

Generated files: [coverage](../figures/accelerator_compilation_coverage.pdf), [overlap](../figures/accelerator_support_overlap.pdf), [family heatmap](../figures/accelerator_family_coverage_heatmap.pdf), and [operator atlas](../figures/accelerator_operator_compatibility_atlas.pdf).

A `compiled` result is positive evidence only for the recorded canonical case. It is not a correctness, performance, arbitrary-shape, or runtime guarantee. Cerebras CS-3 used compile-only mode; GroqFlow converted floating inputs to FP32 before tracing.

In [ ]:
import json
import math
from collections import Counter
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.patches import Circle, FancyBboxPatch, Rectangle

mpl.rcParams.update(
    {
        "figure.dpi": 140,
        "savefig.dpi": 300,
        "font.family": "DejaVu Sans",
        "font.size": 10,
        "axes.titlesize": 16,
        "axes.titleweight": "bold",
        "axes.labelsize": 11,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
    }
)

COLORS = {
    "compiled": "#356AA0",
    "compile_rejected": "#D7654D",
    "needs_fixture": "#B8BDC5",
    "not_applicable": "#F4F4F2",
    "ink": "#20242A",
    "muted": "#66707A",
    "grid": "#D9DDE2",
}
PLATFORM_COLORS = {
    "Graphcore": "#6B4C9A",
    "Cerebras CS-3": "#E07A1F",
    "Groq": "#2B8C8C",
}
PDF_METADATA = {
    "Author": "LASSI-TOOLS compatibility suite",
    "Subject": "Canonical ATen operator compilation coverage",
    "Creator": "Matplotlib",
    "CreationDate": None,
    "ModDate": None,
}

## Load and validate the snapshots

The three snapshots share one Torch-MLIR operator inventory. The common FP16 manifest cell is used for comparison; the Groq checker records that it traces floating tensors as FP32.

In [ ]:
def find_repo_root():
    candidates = [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        if (candidate / "compat_tool" / "snapshots").is_dir():
            return candidate
    raise FileNotFoundError("Run the notebook from inside the LASSI-TOOLS checkout")


REPO_ROOT = find_repo_root()
OUTPUT_DIR = REPO_ROOT / "compat_tool" / "figures"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SNAPSHOT_PATHS = {
    "Graphcore": REPO_ROOT / "compat_tool/snapshots/graphcore-pod64-poptorch/snapshot.json",
    "Cerebras CS-3": REPO_ROOT / "compat_tool/snapshots/alcf-cs3-cerebras-pytorch/snapshot.json",
    "Groq": REPO_ROOT / "compat_tool/snapshots/groq-r01-groqflow/snapshot.json",
}
snapshots = {name: json.loads(path.read_text()) for name, path in SNAPSHOT_PATHS.items()}
platforms = list(SNAPSHOT_PATHS)

operator_sets = [set(snapshot["operators"]) for snapshot in snapshots.values()]
assert all(operators == operator_sets[0] for operators in operator_sets[1:])
all_operators = sorted(operator_sets[0])
included = [
    op
    for op in all_operators
    if all(snapshots[platform]["operators"][op]["included"] for platform in platforms)
]


def status_for(platform, op):
    return snapshots[platform]["operators"][op]["cases"]["fp16"]["canonical"]["result"]["status"]


statuses = {platform: {op: status_for(platform, op) for op in included} for platform in platforms}
runnable = [
    op
    for op in included
    if all(statuses[platform][op] in {"compiled", "compile_rejected"} for platform in platforms)
]

# Fixture and applicability states must be shared for a fair common denominator.
for op in included:
    values = {statuses[platform][op] for platform in platforms}
    if values & {"needs_fixture", "not_applicable"}:
        assert len(values) == 1, (op, values)

inventory_revision = next(iter(snapshots.values()))["source"]["revision"]
inventory_size = len(all_operators)
status_counts = {platform: Counter(statuses[platform].values()) for platform in platforms}

print(f"Inventory revision: {inventory_revision}")
print(f"Inventory: {inventory_size}; included: {len(included)}; runnable: {len(runnable)}")
for platform in platforms:
    compiled = status_counts[platform]["compiled"]
    print(f"{platform:14s} {compiled:3d}/{len(runnable)} = {compiled / len(runnable):.2%}")

## Editable operator-family taxonomy

This is intentionally a compact, name-based publication taxonomy rather than a claim about compiler-internal lowering categories. Rules are evaluated in order. Edit the sets below and rerun the notebook to explore alternative groupings.

In [ ]:
FAMILY_ORDER = [
    "Arithmetic, comparison & logic",
    "Transcendental math",
    "Linear algebra",
    "Complex, FFT & signal",
    "Convolution",
    "Pooling & resampling",
    "Activations",
    "Normalization",
    "Reductions & statistics",
    "Indexing, gather & scatter",
    "Shape, layout & construction",
    "Loss functions",
    "Random & sampling",
    "Quantization",
    "Other",
]

TRANSCENDENTAL = {
    "acos",
    "acosh",
    "asin",
    "asinh",
    "atan",
    "atan2",
    "atanh",
    "cos",
    "cosh",
    "deg2rad",
    "erf",
    "erfinv",
    "exp",
    "exp2",
    "expm1",
    "float_power",
    "ldexp",
    "log",
    "log10",
    "log1p",
    "log2",
    "logaddexp",
    "logaddexp2",
    "logit",
    "pow",
    "rad2deg",
    "reciprocal",
    "rsqrt",
    "sin",
    "sinh",
    "special_expm1",
    "sqrt",
    "square",
    "tan",
    "tanh",
    "xlogy",
}
LINEAR_ALGEBRA = {
    "addbmm",
    "addmm",
    "baddbmm",
    "bilinear",
    "bmm",
    "diag",
    "diag_embed",
    "diagonal",
    "diagonal_copy",
    "dot",
    "einsum",
    "frobenius_norm",
    "linalg_cross",
    "linalg_det",
    "linalg_norm",
    "linalg_qr",
    "linalg_slogdet",
    "linalg_vector_norm",
    "cosine_similarity",
    "linear",
    "matmul",
    "mm",
    "mv",
    "norm",
    "outer",
    "scaled_dot_product_attention",
    "t",
    "t_copy",
    "trace",
    "tril",
    "triu",
}
SIGNAL_COMPLEX = {
    "complex",
    "fft_fft",
    "fft_ifft",
    "fft_rfft",
    "imag",
    "polar",
    "real",
    "stft",
    "view_as_complex",
    "view_as_real",
}
ACTIVATIONS = {
    "celu",
    "dropout",
    "elu",
    "gelu",
    "glu",
    "hardshrink",
    "hardsigmoid",
    "hardswish",
    "hardtanh",
    "leaky_relu",
    "log_sigmoid",
    "log_sigmoid_forward",
    "log_softmax",
    "mish",
    "native_dropout",
    "prelu",
    "relu",
    "relu6",
    "rrelu",
    "rrelu_with_noise",
    "rrelu_with_noise_functional",
    "selu",
    "sigmoid",
    "silu",
    "softmax",
    "softplus",
    "softshrink",
    "threshold",
}
REDUCTIONS = {
    "all",
    "amax",
    "amin",
    "aminmax",
    "any",
    "argmax",
    "argmin",
    "argsort",
    "bincount",
    "count_nonzero",
    "cumprod",
    "cumsum",
    "kthvalue",
    "logcumsumexp",
    "logsumexp",
    "max",
    "mean",
    "min",
    "prod",
    "sort",
    "std",
    "sum",
    "topk",
    "unique_consecutive",
    "unique_dim",
    "var",
    "var_mean",
}
INDEXING = {
    "bucketize",
    "diagonal_scatter",
    "embedding",
    "embedding_bag",
    "gather",
    "index",
    "index_put",
    "index_select",
    "masked_fill",
    "masked_scatter",
    "masked_select",
    "narrow",
    "nonzero",
    "nonzero_numpy",
    "nonzero_static",
    "one_hot",
    "scatter",
    "scatter_add",
    "scatter_reduce",
    "select",
    "select_copy",
    "select_scatter",
    "slice",
    "slice_copy",
    "slice_scatter",
    "where",
}
SHAPE_LAYOUT = {
    "alias",
    "alias_copy",
    "as_strided",
    "as_strided_copy",
    "as_strided_scatter",
    "atleast_1d",
    "atleast_2d",
    "broadcast_tensors",
    "broadcast_to",
    "cat",
    "channel_shuffle",
    "chunk",
    "clone",
    "column_stack",
    "constant_pad_nd",
    "contiguous",
    "copy",
    "cpu",
    "cuda",
    "detach",
    "detach_copy",
    "empty_like",
    "expand",
    "expand_as",
    "expand_copy",
    "fill",
    "flatten",
    "flip",
    "fliplr",
    "flipud",
    "full_like",
    "hstack",
    "lift_fresh_copy",
    "meshgrid",
    "movedim",
    "new_empty",
    "new_empty_strided",
    "new_full",
    "new_ones",
    "new_zeros",
    "numpy_T",
    "ones_like",
    "pad",
    "permute",
    "permute_copy",
    "pixel_shuffle",
    "pixel_unshuffle",
    "reflection_pad1d",
    "reflection_pad2d",
    "reflection_pad3d",
    "repeat",
    "repeat_interleave",
    "replication_pad1d",
    "replication_pad2d",
    "replication_pad3d",
    "reshape",
    "reshape_as",
    "resize",
    "roll",
    "rot90",
    "split",
    "split_copy",
    "split_with_sizes",
    "split_with_sizes_copy",
    "squeeze",
    "squeeze_copy",
    "stack",
    "tensor_split",
    "tile",
    "to",
    "transpose",
    "transpose_copy",
    "type_as",
    "unbind",
    "unbind_copy",
    "unflatten",
    "unfold",
    "unfold_copy",
    "unsqueeze",
    "unsqueeze_copy",
    "view",
    "view_copy",
    "zero",
    "zeros_like",
}
ARITHMETIC_LOGIC = {
    "abs",
    "absolute",
    "add",
    "addcdiv",
    "addcmul",
    "bitwise_and",
    "bitwise_left_shift",
    "bitwise_not",
    "bitwise_or",
    "bitwise_right_shift",
    "bitwise_xor",
    "ceil",
    "clamp",
    "clamp_max",
    "clamp_min",
    "copysign",
    "div",
    "eq",
    "fix",
    "floor",
    "floor_divide",
    "fmax",
    "fmin",
    "fmod",
    "frac",
    "ge",
    "gt",
    "heaviside",
    "isclose",
    "isfinite",
    "isinf",
    "isnan",
    "isneginf",
    "isposinf",
    "le",
    "lerp",
    "logical_and",
    "logical_not",
    "logical_or",
    "logical_xor",
    "lt",
    "max",
    "maximum",
    "min",
    "minimum",
    "mul",
    "nan_to_num",
    "ne",
    "neg",
    "remainder",
    "round",
    "rsub",
    "sgn",
    "sign",
    "signbit",
    "sub",
    "trunc",
}


def base_name(op):
    return op.removeprefix("aten.").split(".")[0]


def classify_family(op):
    base = base_name(op)
    if "conv" in base or base in {"col2im", "im2col"}:
        return "Convolution"
    if "pool" in base or "upsample" in base or base == "grid_sampler":
        return "Pooling & resampling"
    if "norm" in base and base not in LINEAR_ALGEBRA:
        return "Normalization"
    if base in ACTIVATIONS:
        return "Activations"
    if "loss" in base or base in {
        "binary_cross_entropy",
        "binary_cross_entropy_with_logits",
        "cross_entropy_loss",
        "kl_div",
    }:
        return "Loss functions"
    if base in {
        "bernoulli",
        "exponential",
        "multinomial",
        "normal_functional",
        "poisson",
        "rand_like",
        "randn_like",
        "random",
        "uniform",
    }:
        return "Random & sampling"
    if "quantize" in base or base in {"dequantize", "int_repr"}:
        return "Quantization"
    if base in SIGNAL_COMPLEX:
        return "Complex, FFT & signal"
    if base in LINEAR_ALGEBRA:
        return "Linear algebra"
    if base in TRANSCENDENTAL:
        return "Transcendental math"
    if base in REDUCTIONS and op not in {"aten.max.other", "aten.min.other"}:
        return "Reductions & statistics"
    if base in INDEXING:
        return "Indexing, gather & scatter"
    if base in SHAPE_LAYOUT:
        return "Shape, layout & construction"
    if base in ARITHMETIC_LOGIC:
        return "Arithmetic, comparison & logic"
    return "Other"


families = {op: classify_family(op) for op in included}
family_counts = Counter(families.values())
print("Included operators per family:")
for family in FAMILY_ORDER:
    print(f"  {family:32s} {family_counts[family]:3d}")
if family_counts["Other"]:
    print("Other:", [op for op in included if families[op] == "Other"])

## Figure 1 — inventory and overall compilation coverage

In [ ]:
fig = plt.figure(figsize=(10.5, 6.5), layout="constrained")
grid = fig.add_gridspec(2, 1, height_ratios=[1.0, 2.2])
ax_flow = fig.add_subplot(grid[0])
ax_bar = fig.add_subplot(grid[1])

ax_flow.set_xlim(0, 1)
ax_flow.set_ylim(0, 1)
ax_flow.axis("off")
stages = [
    (0.17, inventory_size, "Torch-MLIR inventory"),
    (0.50, len(included), "included forward tensor ops"),
    (0.83, len(runnable), "runnable canonical cases"),
]
for x, count, label in stages:
    box = FancyBboxPatch(
        (x - 0.125, 0.22),
        0.25,
        0.57,
        boxstyle="round,pad=0.016,rounding_size=0.025",
        facecolor="#F5F7FA",
        edgecolor=COLORS["grid"],
        linewidth=1.2,
    )
    ax_flow.add_patch(box)
    ax_flow.text(
        x,
        0.58,
        f"{count}",
        ha="center",
        va="center",
        fontsize=25,
        fontweight="bold",
        color=COLORS["ink"],
    )
    ax_flow.text(x, 0.36, label, ha="center", va="center", fontsize=9.5, color=COLORS["muted"])
for left, right in [(0.295, 0.375), (0.625, 0.705)]:
    ax_flow.annotate(
        "",
        xy=(right, 0.505),
        xytext=(left, 0.505),
        arrowprops={"arrowstyle": "-|>", "lw": 1.4, "color": COLORS["muted"]},
    )

compiled_counts = np.array([status_counts[p]["compiled"] for p in platforms])
rejected_counts = np.array([status_counts[p]["compile_rejected"] for p in platforms])
y = np.arange(len(platforms))
ax_bar.barh(y, compiled_counts, color=COLORS["compiled"], height=0.58, label="Compiled")
ax_bar.barh(
    y,
    rejected_counts,
    left=compiled_counts,
    color=COLORS["compile_rejected"],
    height=0.58,
    label="Compiler rejected",
)
for i, (compiled, rejected) in enumerate(zip(compiled_counts, rejected_counts, strict=True)):
    ax_bar.text(
        compiled / 2,
        i,
        f"{compiled}",
        color="white",
        ha="center",
        va="center",
        fontsize=11,
        fontweight="bold",
    )
    ax_bar.text(
        compiled + rejected / 2,
        i,
        f"{rejected}",
        color="white",
        ha="center",
        va="center",
        fontsize=11,
        fontweight="bold",
    )
    ax_bar.text(
        len(runnable) + 8,
        i,
        f"{compiled / len(runnable):.1%}",
        ha="left",
        va="center",
        fontsize=12,
        fontweight="bold",
        color=COLORS["ink"],
    )
ax_bar.set_yticks(y, platforms)
ax_bar.invert_yaxis()
ax_bar.set_xlim(0, len(runnable) + 48)
ax_bar.set_xlabel(f"Runnable canonical operator cases (n={len(runnable)})")
ax_bar.spines[["top", "right", "left"]].set_visible(False)
ax_bar.tick_params(axis="y", length=0, pad=10)
ax_bar.grid(axis="x", color=COLORS["grid"], linewidth=0.7, alpha=0.8)
ax_bar.set_axisbelow(True)
ax_bar.legend(loc="lower center", bbox_to_anchor=(0.5, -0.36), ncol=2, frameon=False)
fig.suptitle("Canonical ATen operator compilation coverage", color=COLORS["ink"])
fig.text(
    0.5,
    0.005,
    "Shared unresolved coverage: 70 need a valid fixture; 11 are not applicable. "
    "Groq uses an FP32 trace; CS-3 is compile-only.",
    ha="center",
    va="bottom",
    fontsize=8.5,
    color=COLORS["muted"],
)

coverage_path = OUTPUT_DIR / "accelerator_compilation_coverage.pdf"
fig.savefig(
    coverage_path, bbox_inches="tight", metadata={**PDF_METADATA, "Title": fig._suptitle.get_text()}
)
plt.show()
print(coverage_path)

## Figure 2 — three-platform support overlap

This is an exact-count Venn-style diagram. Circle areas are deliberately not presented as proportional.

In [ ]:
def support_pattern(op):
    return tuple(statuses[platform][op] == "compiled" for platform in platforms)


pattern_counts = Counter(support_pattern(op) for op in runnable)
assert sum(pattern_counts.values()) == len(runnable)

# Tuple order: Graphcore, Cerebras CS-3, Groq.
region_positions = {
    (True, False, False): (0.255, 0.62),
    (False, True, False): (0.745, 0.62),
    (False, False, True): (0.50, 0.18),
    (True, True, False): (0.50, 0.73),
    (True, False, True): (0.365, 0.39),
    (False, True, True): (0.635, 0.39),
    (True, True, True): (0.50, 0.53),
}

fig, ax = plt.subplots(figsize=(9.5, 8), layout="constrained")
ax.set_aspect("equal")
ax.set_xlim(0.08, 0.92)
ax.set_ylim(0.02, 0.93)
ax.axis("off")
circles = [
    ("Graphcore", (0.38, 0.59), 0.30),
    ("Cerebras CS-3", (0.62, 0.59), 0.30),
    ("Groq", (0.50, 0.39), 0.30),
]
for platform, center, radius in circles:
    ax.add_patch(
        Circle(
            center,
            radius,
            facecolor=PLATFORM_COLORS[platform],
            edgecolor=PLATFORM_COLORS[platform],
            alpha=0.22,
            linewidth=2.3,
        )
    )

for pattern, position in region_positions.items():
    count = pattern_counts[pattern]
    ax.text(
        *position,
        str(count),
        ha="center",
        va="center",
        fontsize=17,
        fontweight="bold",
        color=COLORS["ink"],
        bbox={
            "boxstyle": "round,pad=0.18",
            "facecolor": "white",
            "edgecolor": "none",
            "alpha": 0.78,
        },
    )

ax.text(
    0.18,
    0.89,
    f"Graphcore\n{status_counts['Graphcore']['compiled']} compiled",
    ha="center",
    va="center",
    fontsize=12,
    fontweight="bold",
    color=PLATFORM_COLORS["Graphcore"],
)
ax.text(
    0.82,
    0.89,
    f"Cerebras CS-3\n{status_counts['Cerebras CS-3']['compiled']} compiled",
    ha="center",
    va="center",
    fontsize=12,
    fontweight="bold",
    color=PLATFORM_COLORS["Cerebras CS-3"],
)
ax.text(
    0.50,
    0.055,
    f"Groq — {status_counts['Groq']['compiled']} compiled",
    ha="center",
    va="center",
    fontsize=12,
    fontweight="bold",
    color=PLATFORM_COLORS["Groq"],
)
none_count = pattern_counts[(False, False, False)]
ax.text(
    0.87,
    0.15,
    f"None\n{none_count}",
    ha="center",
    va="center",
    fontsize=11,
    fontweight="bold",
    color=COLORS["ink"],
    bbox={"boxstyle": "round,pad=0.5", "facecolor": "#F5F7FA", "edgecolor": COLORS["grid"]},
)
ax.set_title(f"Overlap of compiled canonical cases (n={len(runnable)})", pad=12)
fig.text(
    0.5,
    0.015,
    (
        "Exact region counts; circle areas are not proportional. "
        "Groq uses an FP32 trace; CS-3 is compile-only."
    ),
    ha="center",
    fontsize=8.5,
    color=COLORS["muted"],
)

overlap_path = OUTPUT_DIR / "accelerator_support_overlap.pdf"
fig.savefig(overlap_path, bbox_inches="tight", metadata={**PDF_METADATA, "Title": ax.get_title()})
plt.show()
print(overlap_path)

## Figure 3 — operator-family coverage heatmap

In [ ]:
active_families = [
    family for family in FAMILY_ORDER if any(families[op] == family for op in runnable)
]
heatmap_rows = ["All runnable operators", *active_families]
heatmap_counts = np.zeros((len(heatmap_rows), len(platforms)), dtype=int)
heatmap_totals = np.zeros(len(heatmap_rows), dtype=int)

for row, family in enumerate(heatmap_rows):
    ops = runnable if row == 0 else [op for op in runnable if families[op] == family]
    heatmap_totals[row] = len(ops)
    for col, platform in enumerate(platforms):
        heatmap_counts[row, col] = sum(statuses[platform][op] == "compiled" for op in ops)
heatmap_rates = heatmap_counts / heatmap_totals[:, None]

fig_height = 2.3 + 0.53 * len(heatmap_rows)
fig, ax = plt.subplots(figsize=(10.2, fig_height))
fig.subplots_adjust(left=0.34, right=0.86, top=0.92, bottom=0.065)
image = ax.imshow(heatmap_rates, cmap="Blues", vmin=0.25, vmax=1.0, aspect="auto")
ax.set_xticks(np.arange(len(platforms)), platforms)
ax.set_yticks(np.arange(len(heatmap_rows)), heatmap_rows)
ax.tick_params(top=True, bottom=False, labeltop=True, labelbottom=False, length=0)
for row in range(len(heatmap_rows)):
    for col in range(len(platforms)):
        rate = heatmap_rates[row, col]
        color = "white" if rate >= 0.68 else COLORS["ink"]
        ax.text(
            col,
            row,
            f"{rate:.0%}\n{heatmap_counts[row, col]}/{heatmap_totals[row]}",
            ha="center",
            va="center",
            fontsize=9.2,
            fontweight="bold",
            color=color,
        )
ax.set_xticks(np.arange(-0.5, len(platforms), 1), minor=True)
ax.set_yticks(np.arange(-0.5, len(heatmap_rows), 1), minor=True)
ax.grid(which="minor", color="white", linewidth=2.4)
ax.tick_params(which="minor", bottom=False, left=False)
ax.spines[:].set_visible(False)
ax.axhline(0.5, color="white", linewidth=5)
colorbar = fig.colorbar(image, ax=ax, shrink=0.55, pad=0.025)
colorbar.set_label("Compilation coverage")
colorbar.ax.yaxis.set_major_formatter(mpl.ticker.PercentFormatter(1.0))
ax.set_title("Compilation coverage by operator family", pad=18)
fig.text(
    0.5,
    0.015,
    (
        "Cells show compiled / runnable canonical cases. "
        "Name-based families are editable in the notebook. "
        "Groq uses an FP32 trace; CS-3 is compile-only."
    ),
    ha="center",
    fontsize=8.5,
    color=COLORS["muted"],
)

heatmap_path = OUTPUT_DIR / "accelerator_family_coverage_heatmap.pdf"
fig.savefig(heatmap_path, bbox_inches="tight", metadata={**PDF_METADATA, "Title": ax.get_title()})
plt.show()
print(heatmap_path)

## Figure 4 — multipage function-level compatibility atlas

The atlas includes all 437 included functions, including fixture gaps and non-floating cases. It is intended as documentation rather than a single publication panel.

In [ ]:
STATUS_LABELS = {
    "compiled": "C",
    "compile_rejected": "R",
    "needs_fixture": "F",
    "not_applicable": "—",
}
ROWS_PER_PAGE = 34
atlas_path = OUTPUT_DIR / "accelerator_operator_compatibility_atlas.pdf"

with PdfPages(
    atlas_path, metadata={**PDF_METADATA, "Title": "Accelerator operator compatibility atlas"}
) as pdf:
    for family in FAMILY_ORDER:
        family_ops = sorted(op for op in included if families[op] == family)
        if not family_ops:
            continue
        page_count = math.ceil(len(family_ops) / ROWS_PER_PAGE)
        for page_index in range(page_count):
            page_ops = family_ops[page_index * ROWS_PER_PAGE : (page_index + 1) * ROWS_PER_PAGE]
            fig, ax = plt.subplots(figsize=(8.27, 11.69))
            ax.set_xlim(0, 1)
            ax.set_ylim(-1.7, len(page_ops) + 2.8)
            ax.axis("off")
            suffix = f" — page {page_index + 1}/{page_count}" if page_count > 1 else ""
            ax.text(
                0.02,
                len(page_ops) + 2.25,
                family + suffix,
                fontsize=16,
                fontweight="bold",
                color=COLORS["ink"],
                va="center",
            )
            ax.text(
                0.02,
                len(page_ops) + 1.55,
                "Canonical ATen compilation evidence",
                fontsize=9.5,
                color=COLORS["muted"],
            )
            column_x = {"Graphcore": 0.67, "Cerebras CS-3": 0.80, "Groq": 0.93}
            ax.text(0.02, len(page_ops) + 0.70, "Operator", fontsize=9, fontweight="bold")
            for platform, x in column_x.items():
                ax.text(
                    x,
                    len(page_ops) + 0.70,
                    platform.replace("Cerebras CS-3", "CS-3"),
                    ha="center",
                    fontsize=8.5,
                    fontweight="bold",
                    color=PLATFORM_COLORS[platform],
                )
            ax.plot([0.02, 0.98], [len(page_ops) + 0.35] * 2, color=COLORS["grid"], lw=1)

            for row, op in enumerate(page_ops):
                y = len(page_ops) - row - 0.20
                if row % 2 == 0:
                    ax.add_patch(
                        Rectangle(
                            (0.015, y - 0.42), 0.97, 0.84, facecolor="#F7F8FA", edgecolor="none"
                        )
                    )
                ax.text(
                    0.025,
                    y,
                    op,
                    va="center",
                    fontsize=8.1,
                    color=COLORS["ink"],
                    family="DejaVu Sans Mono",
                )
                for platform, x in column_x.items():
                    status = statuses[platform][op]
                    edge = COLORS["muted"] if status == "not_applicable" else COLORS[status]
                    hatch = "///" if status == "not_applicable" else None
                    ax.add_patch(
                        Rectangle(
                            (x - 0.029, y - 0.29),
                            0.058,
                            0.58,
                            facecolor=COLORS[status],
                            edgecolor=edge,
                            linewidth=0.8,
                            hatch=hatch,
                        )
                    )
                    text_color = (
                        "white" if status in {"compiled", "compile_rejected"} else COLORS["ink"]
                    )
                    ax.text(
                        x,
                        y,
                        STATUS_LABELS.get(status, "?"),
                        ha="center",
                        va="center",
                        fontsize=7.4,
                        fontweight="bold",
                        color=text_color,
                    )

            legend_y = -0.72
            legend_items = [
                ("compiled", "C  compiled"),
                ("compile_rejected", "R  compiler rejected"),
                ("needs_fixture", "F  needs fixture"),
                ("not_applicable", "—  not applicable"),
            ]
            for index, (status, label) in enumerate(legend_items):
                x = 0.02 + index * 0.235
                hatch = "///" if status == "not_applicable" else None
                edge = COLORS["muted"] if status == "not_applicable" else COLORS[status]
                ax.add_patch(
                    Rectangle(
                        (x, legend_y - 0.17),
                        0.025,
                        0.34,
                        facecolor=COLORS[status],
                        edgecolor=edge,
                        hatch=hatch,
                    )
                )
                ax.text(
                    x + 0.034, legend_y, label, va="center", fontsize=7.4, color=COLORS["muted"]
                )
            ax.text(
                0.02,
                -1.35,
                (
                    f"Torch-MLIR inventory {inventory_revision[:12]} · "
                    "Groq FP32 trace · CS-3 compile-only"
                ),
                fontsize=7.2,
                color=COLORS["muted"],
            )
            pdf.savefig(fig, bbox_inches="tight")
            plt.close(fig)

print(atlas_path)

## Exported artifacts

In [ ]:
for path in [coverage_path, overlap_path, heatmap_path, atlas_path]:
    print(f"{path.relative_to(REPO_ROOT)}  ({path.stat().st_size / 1024:.1f} KiB)")